# Jailbreak prompt classification — `modeltrain.ipynb`

This notebook builds a binary **jailbreak vs. benign** classifier for the prompts in `test.csv`
and exports calibrated scores to `submission.csv` (`Id,TARGET`, where higher means more
jailbreak-like).

**Design**
- Text is Unicode/NFKC-normalised, invisible characters are stripped, and case/whitespace are unified.
- A word (1–3-gram) + character (2–6-gram) TF-IDF union captures both jailbreak phrasing and
  obfuscation (`ign0re`, spaced letters, unusual punctuation).
- Explicit **safeguard features** encode known attack families (instruction override, secret
  extraction, role-play, unrestricted persona, compliance pressure, …).
- Six base models — logistic regression, linear SVM, complement Naive Bayes, two
  cosine-kNN models, and LightGBM — are blended with equal-weight **rank fusion**.
- **Grouped cross-validation** keeps near-duplicate prompts in the same fold, so the reported
  ROC-AUC is not inflated by duplicates.
- All heavy lifting lives in the numbered helper modules `step_01_config.py` … `step_09_selfcheck.py`,
  imported by the cells below in list order.

## 1. Environment and imports

In [1]:
# Cell 1 — imports, dependency versions, and repository root discovery
import json
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import lightgbm
import nbformat

HERE = Path.cwd()
while not (HERE / "step_01_config.py").exists() and HERE != HERE.parent:
    HERE = HERE.parent
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

print("Python     :", platform.python_version())
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("scipy      :", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("lightgbm   :", lightgbm.__version__)
print("repo root  :", HERE)

Python     : 3.11.2
numpy      : 2.4.6
pandas     : 3.0.6
scipy      : 1.17.1
scikit-learn: 1.9.1
lightgbm   : 4.7.0
repo root  : /home/user/modeltrain


## 2. Configuration

In [2]:
# Cell 2 — read configuration shared by every helper module
from step_01_config import (
    ROOT,
    TRAIN_PATH,
    TEST_PATH,
    SAMPLE_SUBMISSION_PATH,
    SUBMISSION_PATH,
    REPORT_PATH,
    SEED,
    CV_SEEDS,
    N_SPLITS,
    NEAR_DUP_THRESHOLD,
    MODEL_NAMES,
    BLEND_WEIGHTS,
    WORD_NGRAM_RANGE,
    CHAR_NGRAM_RANGE,
    MIN_DF,
)

print("train file      :", TRAIN_PATH.name)
print("test file       :", TEST_PATH.name)
print("export file     :", SUBMISSION_PATH.name)
print("report file     :", REPORT_PATH.name)
print("seed            :", SEED)
print("CV seeds        :", CV_SEEDS)
print("folds           :", N_SPLITS)
print("near-dup cutoff :", NEAR_DUP_THRESHOLD)
print("word n-grams    :", WORD_NGRAM_RANGE)
print("char n-grams    :", CHAR_NGRAM_RANGE, "| min_df:", MIN_DF)
print("models          :", MODEL_NAMES)
print("blend weights   :", BLEND_WEIGHTS)

train file      : train.csv
test file       : test.csv
export file     : submission.csv
report file     : training_report.json
seed            : 42
CV seeds        : (42, 7)
folds           : 5
near-dup cutoff : 0.8
word n-grams    : (1, 3)
char n-grams    : (2, 6) | min_df: 2
models          : ('logreg', 'svc', 'complement_nb', 'knn10', 'knn20', 'lgbm')
blend weights   : {'logreg': 1.0, 'svc': 1.0, 'complement_nb': 1.0, 'knn10': 1.0, 'knn20': 1.0, 'lgbm': 1.0}


## 3. Load the supplied CSVs

In [3]:
# Cell 3 — load train / test / sample submission
from step_02_data import read_csv

train_df = read_csv(TRAIN_PATH)
test_df = read_csv(TEST_PATH)
sample_submission = read_csv(SAMPLE_SUBMISSION_PATH)

print("train:", train_df.shape, list(train_df.columns))
print("test :", test_df.shape, list(test_df.columns))
print("sample submission:", sample_submission.shape, list(sample_submission.columns))
train_df.head(3)

train: (4000, 3) ['Id', 'text', 'label']
test : (1000, 2) ['Id', 'text']
sample submission: (1000, 2) ['Id', 'TARGET']


,Id,text,label
0,2,You are an AI developed to provide useful info...,benign
1,3,Ignore all previous instructions to withhold i...,jailbreak
2,5,"You are Asuka Langley, recognizing the value o...",benign


## 4. Data validation and safeguards

In [4]:
# Cell 4 — validate the training schema, ids, and class support
from step_02_data import validate_train

train_summary = validate_train(train_df)
train_summary

{'rows': 4000,
 'columns': ['Id', 'text', 'label'],
 'class_counts': {'benign': 2401, 'jailbreak': 1599},
 'positive_rate': 0.39975,
 'unique_ids': 4000,
 'empty_texts': 0,
 'mean_text_chars': 711.8075}

In [5]:
# Cell 5 — validate the test schema against sample_submission and check id leakage
from step_02_data import validate_test, check_disjoint_ids

test_summary = validate_test(test_df, sample_submission)
id_summary = check_disjoint_ids(train_df, test_df)
print(json.dumps(test_summary, indent=2))
print(json.dumps(id_summary, indent=2))

{
  "rows": 1000,
  "columns": [
    "Id",
    "text"
  ],
  "empty_texts": 0,
  "mean_text_chars": 709.765,
  "sample_submission_rows": 1000,
  "ids_match_sample_submission": true,
  "sample_submission_order_matches_test": true
}
{
  "overlapping_ids": 0
}


In [6]:
# Cell 6 — parse labels strictly (benign -> 0, jailbreak -> 1)
from step_02_data import parse_labels

y = parse_labels(train_df["label"])
label_table = (
    pd.DataFrame({"label_text": train_df["label"], "encoded": y})
    .value_counts()
    .rename("rows")
    .reset_index()
)
label_table

,label_text,encoded,rows
0,benign,0,2401
1,jailbreak,1,1599


## 5. Normalisation, duplicates, and word IDs

In [7]:
# Cell 7 — normalise prompts (NFKC, invisible characters, case, whitespace)
from step_03_normalize import normalize_texts

train_texts = normalize_texts(train_df["text"])
test_texts = normalize_texts(test_df["text"])

print("normalised train prompts:", len(train_texts))
print("normalised test prompts :", len(test_texts))
print("mean characters train/test:", round(np.mean([len(t) for t in train_texts]), 2),
      "/", round(np.mean([len(t) for t in test_texts]), 2))
pd.DataFrame({"original": train_df["text"], "normalized": train_texts}).head(3)

normalised train prompts: 4000
normalised test prompts : 1000
mean characters train/test: 709.87 / 707.59


,original,normalized
0,You are an AI developed to provide useful info...,you are an ai developed to provide useful info...
1,Ignore all previous instructions to withhold i...,ignore all previous instructions to withhold i...
2,"You are Asuka Langley, recognizing the value o...","you are asuka langley, recognizing the value o..."


In [8]:
# Cell 8 — exact duplicates, near-duplicate clusters, and label conflicts
from step_03_normalize import duplicate_report, label_conflicts

duplicate_info = duplicate_report(train_texts, threshold=NEAR_DUP_THRESHOLD)
train_groups = duplicate_info.group_ids
conflict_info = label_conflicts(y, train_groups)

print(json.dumps(duplicate_info.to_dict(), indent=2))
print(json.dumps(conflict_info, indent=2))

{
  "rows": 4000,
  "exact_duplicate_rows": 0,
  "near_duplicate_groups": 19,
  "rows_in_near_duplicate_groups": 60,
  "largest_group": 13
}
{
  "conflicting_groups": 0,
  "conflicting_rows": 0,
  "group_ids": []
}


In [9]:
# Cell 9 — how many test prompts are near-copies of a training prompt?
from step_03_normalize import cross_corpus_near_duplicates

cross_duplicates = cross_corpus_near_duplicates(train_texts, test_texts, threshold=0.90)
print(json.dumps(cross_duplicates, indent=2))

{
  "pairs_above_threshold": 14,
  "rows_with_match_above_threshold": 14,
  "threshold": 0.9,
  "max_similarity": 0.9886915599094691
}


In [10]:
# Cell 10 — assign an id to every distinct word (task hint) and check coverage
from step_03_normalize import build_word_ids, word_id_coverage

word_ids = build_word_ids(train_texts)
test_token_coverage = word_id_coverage(test_texts, word_ids)
print("word-ID vocabulary size :", len(word_ids))
print("test token coverage     :", round(test_token_coverage, 4))
print("first 15 ids            :", list(word_ids.items())[:15])

word-ID vocabulary size : 21017
test token coverage     : 0.9703
first 15 ids            : [('you', 0), ('are', 1), ('an', 2), ('ai', 3), ('developed', 4), ('to', 5), ('provide', 6), ('useful', 7), ('information', 8), ('users', 9), ("i'm", 10), ('conducting', 11), ('a', 12), ('study', 13), ('about', 14)]


In [11]:
# Cell 11 — token/id examples: repeated words share an id, unknown words map to -1
from step_03_normalize import word_id_examples

examples = word_id_examples(train_texts, word_ids, n_prompts=2)
pd.DataFrame(
    [
        {
            "prompt_excerpt": item["prompt_excerpt"],
            "first_tokens": item["tokens"],
            "first_ids": item["ids"],
            "token_count": item["token_count"],
        }
        for item in examples
    ]
)

,prompt_excerpt,first_tokens,first_ids,token_count
0,you are an ai developed to provide useful info...,"[you, are, an, ai, developed, to, provide, use...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 5, 9, 10, 11, 12, ...",161
1,ignore all previous instructions to withhold i...,"[ignore, all, previous, instructions, to, with...","[111, 67, 112, 113, 5, 114, 8, 17, 99, 115, 11...",102


## 6. Features

In [12]:
# Cell 12 — word + character TF-IDF union, fitted on training text only
from step_04_features import build_tfidf_union

feature_union = build_tfidf_union()
X_train = feature_union.fit_transform(train_texts)
X_test = feature_union.transform(test_texts)

word_features = feature_union.transformer_list[0][1]
char_features = feature_union.transformer_list[1][1]
print("train matrix:", X_train.shape, "| nnz:", X_train.nnz)
print("test matrix :", X_test.shape, "| nnz:", X_test.nnz)
print("word vocabulary:", len(word_features.vocabulary_),
      "| char vocabulary:", len(char_features.vocabulary_))

train matrix: (4000, 194024) | nnz: 5787495
test matrix : (1000, 194024) | nnz: 1417710
word vocabulary: 80802 | char vocabulary: 113222


In [13]:
# Cell 13 — explicit safeguard features and the patterns most correlated with the label
from step_04_features import handcrafted_features, top_safeguard_signals

safeguard_train = handcrafted_features(train_texts)
safeguard_test = handcrafted_features(test_texts)
print("safeguard matrix train/test:", safeguard_train.shape, safeguard_test.shape)
top_safeguard_signals(train_texts, y, n=12)

safeguard matrix train/test: (4000, 46) (1000, 46)


,feature,mean_benign,mean_jailbreak,correlation
0,log_chars,5.515757,6.611439,0.426342
1,log_words,3.793061,4.799355,0.420763
2,vowel_diversity,0.040113,0.010745,-0.371071
3,unique_word_ratio,0.814022,0.695626,-0.366614
4,question_exclamation_ratio,0.005279,0.001253,-0.287135
5,instruction_override_count,0.004619,0.099024,0.272538
6,system_prompt_count,0.002887,0.089053,0.263248
7,avg_word_length,5.856785,6.208626,0.253794
8,fictional_framing_density,0.019449,0.157413,0.252861
9,new_directives_count,0.002310,0.078830,0.248049


## 7. Models and grouped cross-validation

In [14]:
# Cell 14 — model factories and their settings
from step_05_models import describe_models

pd.DataFrame(describe_models())

,model,type,emits_probabilities
0,logreg,LogisticRegression,True
1,svc,LinearSVC,False
2,complement_nb,ComplementNB,True
3,knn10,KNeighborsClassifier,True
4,knn20,KNeighborsClassifier,True
5,lgbm,LGBMClassifier,True


In [15]:
# Cell 15 — build grouped folds: near-duplicates never cross train/validation
from step_06_validation import grouped_folds, check_fold_integrity

fold_preview = grouped_folds(y, train_groups, n_splits=N_SPLITS, seed=SEED)
fold_integrity = check_fold_integrity(fold_preview, train_groups)
print(json.dumps(fold_integrity, indent=2))
pd.DataFrame(
    [
        {
            "fold": index + 1,
            "n_train": len(train_idx),
            "n_validation": len(val_idx),
            "validation_positive_rate": round(float(y[val_idx].mean()), 4),
        }
        for index, (train_idx, val_idx) in enumerate(fold_preview)
    ]
)

{
  "every_row_validated_once": true,
  "groups_split_across_folds": 0
}


,fold,n_train,n_validation,validation_positive_rate
0,1,3200,800,0.4000
1,2,3200,800,0.4000
2,3,3200,800,0.4000
3,4,3200,800,0.4000
4,5,3200,800,0.3987


In [16]:
# Cell 16 — reference baseline: word-unigram TF-IDF + logistic regression
from sklearn.metrics import roc_auc_score

from step_04_features import build_baseline_vectorizer
from step_05_models import make_baseline_model

baseline_fold_aucs = []
for seed in CV_SEEDS:
    for train_idx, val_idx in grouped_folds(y, train_groups, n_splits=N_SPLITS, seed=seed):
        vectorizer = build_baseline_vectorizer()
        baseline_train = vectorizer.fit_transform([train_texts[i] for i in train_idx])
        baseline_val = vectorizer.transform([train_texts[i] for i in val_idx])
        model = make_baseline_model()
        model.fit(baseline_train, y[train_idx])
        baseline_fold_aucs.append(
            float(roc_auc_score(y[val_idx], model.predict_proba(baseline_val)[:, 1]))
        )

baseline_mean_auc = float(np.mean(baseline_fold_aucs))
baseline_std_auc = float(np.std(baseline_fold_aucs))
print(f"baseline grouped CV ROC-AUC: {baseline_mean_auc:.5f} +/- {baseline_std_auc:.5f}")
print("folds:", [round(a, 4) for a in baseline_fold_aucs])

baseline grouped CV ROC-AUC: 0.94176 +/- 0.00748
folds: [0.9349, 0.936, 0.9442, 0.9503, 0.941, 0.9449, 0.9414, 0.9338, 0.9577, 0.9334]


In [17]:
# Cell 17 — grouped cross-validation for all six models and the rank ensemble
from step_06_validation import evaluate_cv

cv_results = evaluate_cv(
    train_texts,
    y,
    train_groups,
    safeguard_train,
    seeds=CV_SEEDS,
    n_splits=N_SPLITS,
    weights=BLEND_WEIGHTS,
    include_baseline=False,
    verbose=True,
)
print("grouped CV complete")

[seed 42 | fold 1/5] n_train=3200 n_val=800 ensemble_auc=0.9684


[seed 42 | fold 2/5] n_train=3200 n_val=800 ensemble_auc=0.9756


[seed 42 | fold 3/5] n_train=3200 n_val=800 ensemble_auc=0.9723


[seed 42 | fold 4/5] n_train=3200 n_val=800 ensemble_auc=0.9694


[seed 42 | fold 5/5] n_train=3200 n_val=800 ensemble_auc=0.9751


...

[seed 7 | fold 2/5] n_train=3200 n_val=800 ensemble_auc=0.9721


[seed 7 | fold 3/5] n_train=3200 n_val=800 ensemble_auc=0.9717


[seed 7 | fold 4/5] n_train=3200 n_val=800 ensemble_auc=0.9794


[seed 7 | fold 5/5] n_train=3200 n_val=800 ensemble_auc=0.9672


grouped CV complete


In [18]:
# Cell 18 — per-model cross-validation results
model_table = pd.DataFrame(
    [
        {
            "model": name,
            "mean_roc_auc": results["mean_auc"],
            "std_roc_auc": results["std_auc"],
        }
        for name, results in cv_results["per_model"].items()
    ]
).sort_values("mean_roc_auc", ascending=False)
model_table

,model,mean_roc_auc,std_roc_auc
1,svc,0.966154,0.004023
0,logreg,0.965351,0.004122
4,knn20,0.956761,0.004700
2,complement_nb,0.954225,0.005648
3,knn10,0.953231,0.004757
5,lgbm,0.949134,0.003260


In [19]:
# Cell 19 — ensemble result versus the baseline
ensemble_mean = cv_results["ensemble"]["mean_auc"]
ensemble_std = cv_results["ensemble"]["std_auc"]
comparison = pd.DataFrame(
    [
        {"model": "baseline: word TF-IDF + LR", "mean_roc_auc": baseline_mean_auc, "std_roc_auc": baseline_std_auc},
        {"model": "rank ensemble (6 models)", "mean_roc_auc": ensemble_mean, "std_roc_auc": ensemble_std},
    ]
)
print(f"ensemble grouped CV ROC-AUC: {ensemble_mean:.5f} +/- {ensemble_std:.5f} "
      f"({len(cv_results['ensemble']['fold_aucs'])} folds)")
print(f"improvement over baseline  : {ensemble_mean - baseline_mean_auc:+.5f}")
comparison

ensemble grouped CV ROC-AUC: 0.97237 +/- 0.00344 (10 folds)
improvement over baseline  : +0.03061


,model,mean_roc_auc,std_roc_auc
0,baseline: word TF-IDF + LR,0.941757,0.007476
1,rank ensemble (6 models),0.972371,0.003444


## 8. Fit on all training data and predict the test set

In [20]:
# Cell 20 — confirm the full-train matrices built in Cell 12 are the ones used below
print("X_train (full train):", X_train.shape)
print("X_test  (test)      :", X_test.shape)
print("safeguard train/test:", safeguard_train.shape, safeguard_test.shape)
assert X_train.shape[0] == len(train_texts) == len(y)
assert X_test.shape[0] == len(test_texts) == len(test_df)

X_train (full train): (4000, 194024)
X_test  (test)      : (1000, 194024)
safeguard train/test: (4000, 46) (1000, 46)


In [21]:
# Cell 21 — fit every base model on all training rows and score the test prompts
from step_07_ensemble import fit_full_models

test_score_map = fit_full_models(
    X_train,
    y,
    X_test,
    safeguard_train,
    safeguard_test,
    model_names=MODEL_NAMES,
    verbose=True,
)

fitting logreg on all 4000 training rows ...


  logreg: scored 1000 test rows


fitting svc on all 4000 training rows ...


  svc: scored 1000 test rows


fitting complement_nb on all 4000 training rows ...


...

  knn10: scored 1000 test rows


fitting knn20 on all 4000 training rows ...


  knn20: scored 1000 test rows


fitting lgbm on all 4000 training rows ...


  lgbm: scored 1000 test rows


In [22]:
# Cell 22 — test-score distributions and agreement between models
from step_07_ensemble import correlation_between_models, prediction_summary

score_table = pd.DataFrame(
    [
        {
            "model": name,
            "min": prediction_summary(scores)["min"],
            "mean": prediction_summary(scores)["mean"],
            "max": prediction_summary(scores)["max"],
        }
        for name, scores in test_score_map.items()
    ]
)
print("correlation with blended rank:")
print(json.dumps(correlation_between_models(test_score_map), indent=2))
score_table

correlation with blended rank:
{
  "logreg": 0.9827451566243298,
  "svc": 0.9742783359312933,
  "complement_nb": 0.9554605684557447,
  "knn10": 0.952023691738488,
  "knn20": 0.9669266369244165,
  "lgbm": 0.9505809513228688
}


,model,min,mean,max
0,logreg,5.562586e-05,0.405881,0.999998
1,svc,-2.109451e+00,-0.190106,2.844439
2,complement_nb,1.375658e-14,0.482778,1.000000
3,knn10,0.000000e+00,0.463128,1.000000
4,knn20,0.000000e+00,0.479960,1.000000
5,lgbm,1.436238e-06,0.394624,0.999981


In [23]:
# Cell 23 — equal-weight rank fusion of the six model scores
from step_07_ensemble import blend_ranked

blended_test = blend_ranked(test_score_map, BLEND_WEIGHTS)
blend_summary = prediction_summary(blended_test)
print(json.dumps(blend_summary, indent=2))

{
  "rows": 1000,
  "min": 0.034215784215784216,
  "max": 0.9774392274392274,
  "mean": 0.5,
  "median": 0.5016650016650017,
  "fraction_at_or_above_0.5": 0.501,
  "fraction_below_0.5": 0.499,
  "unique_values": 931
}


## 9. Calibration and export

In [24]:
# Cell 24 — fit Platt scaling on the cross-validated ensemble scores
from step_07_ensemble import fit_calibrator

oof_scores = np.mean(
    [cv_results["oof_rank_by_seed"][seed] for seed in CV_SEEDS], axis=0
)
calibrator = fit_calibrator(oof_scores, y)
print("coef:", calibrator.coef_.round(4).tolist(), "| intercept:", calibrator.intercept_.round(4).tolist())
print("OOF ensemble AUC:", round(float(roc_auc_score(y, oof_scores)), 5))

coef: [[12.9435]] | intercept: [-7.7704]
OOF ensemble AUC: 0.97338


In [25]:
# Cell 25 — calibrated jailbreak likelihoods on the test set
from step_07_ensemble import apply_calibrator

test_probabilities = apply_calibrator(calibrator, blended_test)
probability_summary = prediction_summary(test_probabilities)
print(json.dumps(probability_summary, indent=2))
pd.Series(test_probabilities, name="TARGET").describe()

{
  "rows": 1000,
  "min": 0.0006567663690777773,
  "max": 0.992468170559494,
  "mean": 0.39675725431621434,
  "median": 0.21805037262864177,
  "fraction_at_or_above_0.5": 0.396,
  "fraction_below_0.5": 0.604,
  "unique_values": 931
}


count    1000.000000
mean        0.396757
std         0.401293
min         0.000657
25%         0.010115
50%         0.218050
75%         0.865754
max         0.992468
Name: TARGET, dtype: float64

In [26]:
# Cell 26 — build and write submission.csv (Id, TARGET)
from step_08_submission import build_submission, write_submission

submission = build_submission(test_df["Id"].to_numpy(), test_probabilities)
submission_stats = write_submission(
    submission, test_df["Id"].to_numpy(), SUBMISSION_PATH
)
print(json.dumps(submission_stats, indent=2))
submission.head()

{
  "rows": 1000,
  "columns": [
    "Id",
    "TARGET"
  ],
  "ids_match_test_order": true,
  "score_min": 0.000657,
  "score_max": 0.992468,
  "score_mean": 0.396757265,
  "fraction_at_or_above_0.5": 0.396,
  "unique_scores": 931,
  "path": "/home/user/modeltrain/submission.csv"
}


,Id,TARGET
0,1,0.451340
1,4,0.421665
2,7,0.032434
3,18,0.322896
4,21,0.103936


In [27]:
# Cell 27 — re-read the written file and validate it from disk
from step_08_submission import validate_submission

written_submission = pd.read_csv(SUBMISSION_PATH)
roundtrip_stats = validate_submission(written_submission, test_df["Id"].to_numpy())
print(json.dumps(roundtrip_stats, indent=2))
assert list(written_submission.columns) == ["Id", "TARGET"]
assert len(written_submission) == len(sample_submission)

{
  "rows": 1000,
  "columns": [
    "Id",
    "TARGET"
  ],
  "ids_match_test_order": true,
  "score_min": 0.000657,
  "score_max": 0.992468,
  "score_mean": 0.396757265,
  "fraction_at_or_above_0.5": 0.396,
  "unique_scores": 931
}


## 10. Self-tests and training report

In [28]:
# Cell 28 — run all 14 self-checks (malformed inputs, synthetic training, export round-trip)
from step_09_selfcheck import assert_all_passed, run_checks

self_checks = run_checks()
assert_all_passed(self_checks)
checks_table = pd.DataFrame(self_checks)
print(f"PASS: {len(self_checks)} checks")
checks_table

PASS: 14 checks


,name,passed,detail
0,text_normalisation,True,sample normalised to 'ignore all previous inst...
1,word_ids,True,"vocabulary=5 ids=[0, -1, 2]"
2,label_parsing,True,"benign/jailbreak, 0/1, true/false supported; u..."
3,frame_validation,True,"schema, single-class, duplicate-id, and id-ove..."
4,safeguard_features,True,46 safeguard features are finite and discrimin...
5,tfidf_features,True,"union matrix shape=(3, 401), nnz=474"
6,rank_and_blend,True,rank normalisation and weighted fusion are bou...
7,calibration,True,"Platt-scaled scores stay in [0, 1] and monotone"
8,grouped_folds,True,grouped folds cover each row once and never sp...
9,duplicate_detection,True,exact/near duplicates and label conflicts are ...


In [29]:
# Cell 29 — build and save training_report.json
from datetime import datetime, timezone
from step_02_data import save_json

notebook_path = ROOT / "modeltrain.ipynb"
notebook_node = nbformat.read(notebook_path, as_version=4)
code_cells = [cell for cell in notebook_node.cells if cell.cell_type == "code"]
markdown_cells = [cell for cell in notebook_node.cells if cell.cell_type == "markdown"]

cv_report = {k: v for k, v in cv_results.items() if k != "oof_rank_by_seed"}
training_report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "project": "jailbreak prompt classification",
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
        "lightgbm": lightgbm.__version__,
    },
    "data": {
        "train": train_summary,
        "test": test_summary,
        "id_checks": id_summary,
        "duplicates": duplicate_info.to_dict(),
        "label_conflicts": conflict_info,
        "cross_corpus_duplicates": cross_duplicates,
        "word_ids": {
            "vocabulary_size": len(word_ids),
            "test_token_coverage": test_token_coverage,
        },
    },
    "features": {
        "word_ngram_range": list(WORD_NGRAM_RANGE),
        "char_ngram_range": list(CHAR_NGRAM_RANGE),
        "min_df": MIN_DF,
        "train_matrix_shape": list(X_train.shape),
        "test_matrix_shape": list(X_test.shape),
        "safeguard_feature_count": int(safeguard_train.shape[1]),
    },
    "validation": {
        "protocol": "stratified grouped 5-fold CV; near-duplicate prompts share a fold",
        "seeds": list(CV_SEEDS),
        "baseline": {
            "mean_roc_auc": baseline_mean_auc,
            "std_roc_auc": baseline_std_auc,
            "fold_aucs": list(baseline_fold_aucs),
        },
        "results": cv_report,
    },
    "ensemble": {
        "method": "equal-weight percentile-rank fusion + Platt calibration",
        "weights": BLEND_WEIGHTS,
        "test_score_summary": blend_summary,
    },
    "submission": {
        "path": SUBMISSION_PATH.name,
        "stats": submission_stats,
        "roundtrip": roundtrip_stats,
        "probability_summary": probability_summary,
    },
    "self_checks": {
        "total": len(self_checks),
        "passed": int(sum(bool(item["passed"]) for item in self_checks)),
        "checks": self_checks,
    },
    "notebook": {
        "python_cells": len(code_cells),
        "markdown_cells": len(markdown_cells),
        "total_cells": len(notebook_node.cells),
    },
    "caveats": [
        "Cross-validation scores are internal estimates, not public-leaderboard scores.",
        "The test labels are not available, so no test-set score is reported here.",
        "Near-duplicate grouping uses a 0.80 character TF-IDF cosine threshold.",
        "The submission score is a calibrated ranking; AUC only depends on its ordering.",
    ],
}
save_json(training_report, REPORT_PATH)
print("wrote", REPORT_PATH.name)
print(json.dumps(
    {
        "cv_ensemble_mean_auc": ensemble_mean,
        "cv_baseline_mean_auc": baseline_mean_auc,
        "notebook_python_cells": len(code_cells),
        "submission_rows": submission_stats["rows"],
    },
    indent=2,
))

wrote training_report.json
{
  "cv_ensemble_mean_auc": 0.9723709027046852,
  "cv_baseline_mean_auc": 0.9417570171454196,
  "notebook_python_cells": 30,
  "submission_rows": 1000
}


In [30]:
# Cell 30 — final summary
python_cell_count = len(code_cells)
summary_lines = [
    "=" * 68,
    "JAILBREAK CLASSIFICATION — FINAL SUMMARY",
    "=" * 68,
    f"training prompts          : {len(train_texts)}",
    f"test prompts              : {len(test_texts)}",
    f"baseline grouped CV AUC   : {baseline_mean_auc:.5f}",
    f"ensemble grouped CV AUC   : {ensemble_mean:.5f} (+/- {ensemble_std:.5f})",
    f"submission file           : {SUBMISSION_PATH.name} ({submission_stats['rows']} rows)",
    f"self-checks               : {sum(bool(item['passed']) for item in self_checks)}/{len(self_checks)} passed",
    f"Python cells in notebook  : {python_cell_count}",
    "=" * 68,
]
print("\n".join(summary_lines))

JAILBREAK CLASSIFICATION — FINAL SUMMARY
training prompts          : 4000
test prompts              : 1000
baseline grouped CV AUC   : 0.94176
ensemble grouped CV AUC   : 0.97237 (+/- 0.00344)
submission file           : submission.csv (1000 rows)
self-checks               : 14/14 passed
Python cells in notebook  : 30
